# Startup Profit Prediction

## Project Overview

This project focuses on predicting the profit of startup companies using historical business metrics. By analyzing expenditures across Research & Development (R&D), Administration, and Marketing, along with the state of operation, we aim to build regression models that accurately estimate profit. We implement and compare multiple regression algorithms, specifically Linear Regression, Lasso Regression, and Ridge Regression, to determine the most effective approach.

## Import Required Libraries

Before starting the data analysis and modeling, we need to load the necessary Python libraries. We import `pandas` for data manipulation, `numpy` for mathematical operations, and scikit-learn utilities for data splitting, model creation, and performance metrics. These tools form the core foundation of our machine learning pipeline.

In [1]:
import pandas as pd

## Load Dataset

We load the startup dataset from the local `50_Startups.csv` file using pandas. This step imports the raw records into a DataFrame structure so we can begin exploring the data layout and summary statistics. Initial inspection of rows helps us understand the types of features and target variable we are working with.

In [2]:
data=pd.read_csv('50_Startups.csv')
data.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


## Exploratory Data Analysis

Exploratory Data Analysis (EDA) is performed to inspect the dataset properties, dimensions, column names, and data types. We call methods like `.shape`, `.columns`, `.dtypes`, `.info()`, and `.describe()` to get a statistical summary of the variables. This step ensures we understand the scale of the features and identify whether there are any missing values or anomalies in the dataset.

In [3]:
data.shape

(50, 5)

In [ ]:
data.columns

Index(['R&D Spend', 'Administration', 'Marketing Spend', 'State', 'Profit'], dtype='object')

In [4]:
data.dtypes

,0
R&D Spend,float64
Administration,float64
Marketing Spend,float64
State,object
Profit,float64


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [6]:
data.describe()

,R&D Spend,Administration,Marketing Spend,Profit
count,50.000000,50.000000,50.000000,50.000000
mean,73721.615600,121344.639600,211025.097800,112012.639200
std,45902.256482,28017.802755,122290.310726,40306.180338
min,0.000000,51283.140000,0.000000,14681.400000
25%,39936.370000,103730.875000,129300.132500,90138.902500
50%,73051.080000,122699.795000,212716.240000,107978.190000
75%,101602.800000,144842.180000,299469.085000,139765.977500
max,165349.200000,182645.560000,471784.100000,192261.830000


## Data Preprocessing

Data preprocessing is vital to handle missing values and check categorical distributions before feeding the variables to the model. We verify that there are no null values across our columns using `.isnull().sum()`. Additionally, we check the unique states represented in the categorical `State` column to prepare for feature encoding.

In [7]:
data.isnull().sum()   #to check for any null/missing values.

,0
R&D Spend,0
Administration,0
Marketing Spend,0
State,0
Profit,0


In [8]:
data.isnull().sum().sum()

np.int64(0)

In [9]:
data['State'].nunique()

3

In [10]:
data['State'].unique()

array(['New York', 'California', 'Florida'], dtype=object)

## Feature Engineering

Since machine learning models require numerical representation, we convert the categorical `State` column into dummy indicator variables using one-hot encoding. We then concatenate these dummy variables back with our main dataset and drop the original text-based `State` column. Finally, we convert the entire dataset to integer format to maintain compatibility and prevent formatting exceptions during modeling.

In [11]:
# converting State column which is object datatype to int data type.
columns=['State']
data1=data[columns]
dummies=pd.get_dummies(data1,columns=['State'])
dummies

,State_California,State_Florida,State_New York
0,False,False,True
1,True,False,False
2,False,True,False
3,False,False,True
4,False,True,False
5,False,False,True
6,True,False,False
7,False,True,False
8,False,False,True
9,True,False,False


In [12]:
mergeddata= pd.concat([data,dummies],axis='columns')
mergeddata

,R&D Spend,Administration,Marketing Spend,State,Profit,State_California,State_Florida,State_New York
0,165349.20,136897.80,471784.10,New York,192261.83,False,False,True
1,162597.70,151377.59,443898.53,California,191792.06,True,False,False
2,153441.51,101145.55,407934.54,Florida,191050.39,False,True,False
3,144372.41,118671.85,383199.62,New York,182901.99,False,False,True
4,142107.34,91391.77,366168.42,Florida,166187.94,False,True,False
5,131876.90,99814.71,362861.36,New York,156991.12,False,False,True
6,134615.46,147198.87,127716.82,California,156122.51,True,False,False
7,130298.13,145530.06,323876.68,Florida,155752.60,False,True,False
8,120542.52,148718.95,311613.29,New York,152211.77,False,False,True
9,123334.88,108679.17,304981.62,California,149759.96,True,False,False


In [13]:
newdata=mergeddata.drop(['State'],axis='columns')
newdata

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,False,True
1,162597.70,151377.59,443898.53,191792.06,True,False,False
2,153441.51,101145.55,407934.54,191050.39,False,True,False
3,144372.41,118671.85,383199.62,182901.99,False,False,True
4,142107.34,91391.77,366168.42,166187.94,False,True,False
5,131876.90,99814.71,362861.36,156991.12,False,False,True
6,134615.46,147198.87,127716.82,156122.51,True,False,False
7,130298.13,145530.06,323876.68,155752.60,False,True,False
8,120542.52,148718.95,311613.29,152211.77,False,False,True
9,123334.88,108679.17,304981.62,149759.96,True,False,False


In [14]:
#converting data into int datatype to avoid errors below.
prepareddata=newdata.astype(int)
prepareddata.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
0,165349,136897,471784,192261,0,0,1
1,162597,151377,443898,191792,1,0,0
2,153441,101145,407934,191050,0,1,0
3,144372,118671,383199,182901,0,0,1
4,142107,91391,366168,166187,0,1,0


In [15]:
prepareddata.columns

Index(['R&D Spend', 'Administration', 'Marketing Spend', 'Profit',
       'State_California', 'State_Florida', 'State_New York'],
      dtype='object')

In [16]:
prepareddata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   R&D Spend         50 non-null     int64
 1   Administration    50 non-null     int64
 2   Marketing Spend   50 non-null     int64
 3   Profit            50 non-null     int64
 4   State_California  50 non-null     int64
 5   State_Florida     50 non-null     int64
 6   State_New York    50 non-null     int64
dtypes: int64(7)
memory usage: 2.9 KB


## Train-Test Split

To evaluate our models' generalization performance, we partition our data into features (independent variables `X`) and target (dependent variable `y`). We split the features and target arrays into training and testing sets using an 80/20 ratio. The training set is used to optimize the model coefficients, while the test set serves as unseen data for evaluation.

In [17]:
# Import train_test_split from sklearn.model_selection
from sklearn.model_selection import train_test_split
# Here, X is the data which will have features and y will have our target.
x=prepareddata[['R&D Spend', 'Administration', 'Marketing Spend','State_California', 'State_Florida', 'State_New York']]
y=prepareddata['Profit']

In [18]:
# Split data into training data and testing data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
#Ratio used for splitting training and testing data is 8:2 respectively

## Linear Regression

We initialize and train a standard Ordinary Least Squares Linear Regression model. This algorithm estimates linear coefficients for each feature to predict profit, minimizing the residual sum of squares between observed and predicted targets. We fit the training data and then compute predictions on the test set.

In [19]:
# Importing linear regression model
from sklearn.linear_model import LinearRegression
reg1 = LinearRegression()

In [20]:
# Fitting data into the model.
reg1.fit(x_train, y_train)

LinearRegression()

In [21]:
# Making predictions
pred1 = reg1.predict(x_test)

In [22]:
pred1

array([ 89607.253587  , 195180.87366165,  45174.04824715, 147079.16688442,
        96549.83536077, 162035.46771956, 117292.31078843, 115714.90153417,
       101296.78308717, 110998.61459082])

## Lasso Regression

Lasso (Least Absolute Shrinkage and Selection Operator) Regression is created as a regularized version of linear regression. It applies L1 regularization, which penalizes the absolute size of the coefficients and can shrink some coefficients to zero, effectively performing feature selection. We fit the Lasso model on our training data and predict on the test split.

In [23]:
# Importing model
from sklearn.linear_model import Lasso
reg2 = Lasso()

In [24]:
# Fitting data into the model.
reg2.fit(x_train, y_train)

Lasso()

In [25]:
# Making predictions
pred2 = reg2.predict(x_test)

In [26]:
pred2

array([ 89607.52130095, 195184.61551644,  45176.59323854, 147082.31374333,
        96552.46375205, 162032.22887255, 117292.75754307, 115711.80260624,
       101292.93058268, 110995.56774413])

## Ridge Regression

Ridge Regression adds L2 regularization to prevent overfitting by penalizing the square of the coefficients. This shrinkage method helps manage multi-collinearity among features and produces more stable parameter estimations. After initializing and fitting the Ridge regression model, we predict on the test set.

In [27]:
# Importing model
from sklearn.linear_model import Ridge
reg3 = Ridge()

In [28]:
# Fitting data into the model.
reg3.fit(x_train, y_train)

Ridge()

In [29]:
# Making predictions
pred3= reg3.predict(x_test)

In [30]:
pred3

array([ 89613.16565207, 195212.08410713,  45193.42070085, 147104.72782167,
        96569.89919054, 162006.17848773, 117299.2648746 , 115686.96882819,
       101260.96391285, 110971.35809661])

## Model Evaluation

To compare the performance of our three trained models, we evaluate their accuracy ($R^2$ score) on the training set and compute the Root Mean Squared Error (RMSE) on the testing set. Training accuracy measures the percentage of variance explained, while RMSE quantifies prediction error in actual profit values. Reviewing these metrics helps select the optimal model.

In [31]:
import numpy as np
from sklearn.metrics import mean_squared_error
print("Model\t\t\t RootMeanSquareError \t\t Accuracy of the model")
print("""Linear Regression \t\t {:.4f} \t \t\t {:.4f}""".format(  np.sqrt(mean_squared_error(y_test, pred1)), reg1.score(x_train,y_train)))
print("""Lasso Regression \t\t {:.4f} \t \t\t {:.4f}""".format(  np.sqrt(mean_squared_error(y_test, pred2)), reg2.score(x_train,y_train)))
print("""Ridge Regression \t\t {:.4f} \t \t\t {:.4f}""".format(  np.sqrt(mean_squared_error(y_test, pred3)), reg3.score(x_train,y_train)))

Model			 RootMeanSquareError 		 Accuracy of the model
Linear Regression 		 8791.5072 	 		 0.9522
Lasso Regression 		 8791.2274 	 		 0.9522
Ridge Regression 		 8789.1956 	 		 0.9522


## Model Serialization using Joblib

For ease of deployment and future reuse, we serialize the trained estimators using the Joblib python library. Model serialization saves the fitted scikit-learn objects into binary files on disk, avoiding the need to retrain them on next run. We save each algorithm's trained estimator under its respective file path: `linear_regression.joblib`, `lasso_regression.joblib`, and `ridge_regression.joblib`.

In [32]:
import joblib

# Save the trained models in joblib format
joblib.dump(reg1, 'linear_regression.joblib')
joblib.dump(reg2, 'lasso_regression.joblib')
joblib.dump(reg3, 'ridge_regression.joblib')
print("Models successfully serialized using Joblib!")

Models successfully serialized using Joblib!


## Loading Saved Models

To demonstrate the deployment workflow, we reload the saved model files from the disk back into memory. Using `joblib.load()`, we recreate scikit-learn models that retain all their coefficients and preprocessing configurations. This verifies that our saved binaries are fully functional and ready for prediction.

In [33]:
# Load each saved model from disk
loaded_lr = joblib.load('linear_regression.joblib')
loaded_lasso = joblib.load('lasso_regression.joblib')
loaded_ridge = joblib.load('ridge_regression.joblib')
print("Saved models successfully loaded from disk!")

Saved models successfully loaded from disk!


## Making Predictions using Saved Models

We test our reloaded models by inputting a sample startup record and performing inference. The sample represents a company with specific R&D, Administration, and Marketing expenditures in a specific state. We run predictions on all three loaded models and print their estimates side-by-side to verify consistency. This ensures that model restoration does not affect prediction accuracy.

In [34]:
# Define one sample startup input matching the training features
# Values used: R&D Spend=165349, Administration=136897, Marketing Spend=471784, State=New York (California=0, Florida=0, New York=1)
sample_data = pd.DataFrame([[165349, 136897, 471784, 0, 0, 1]],
                            columns=['R&D Spend', 'Administration', 'Marketing Spend', 'State_California', 'State_Florida', 'State_New York'])

# Predict using each loaded model
pred_lr = loaded_lr.predict(sample_data)[0]
pred_lasso = loaded_lasso.predict(sample_data)[0]
pred_ridge = loaded_ridge.predict(sample_data)[0]

print("Sample Startup Input Data:")
print(sample_data.to_string(index=False))
print("-" * 50)
print(f"Loaded Linear Regression Prediction:  ${pred_lr:,.2f}")
print(f"Loaded Lasso Regression Prediction:   ${pred_lasso:,.2f}")
print(f"Loaded Ridge Regression Prediction:   ${pred_ridge:,.2f}")

Sample Startup Input Data:
 R&D Spend  Administration  Marketing Spend  State_California  State_Florida  State_New York
    165349          136897           471784                 0              0               1
--------------------------------------------------
Loaded Linear Regression Prediction:  $195,180.87
Loaded Lasso Regression Prediction:   $195,184.62
Loaded Ridge Regression Prediction:   $195,212.08


## Conclusion

* All 3 regression algorithms used in this project are equally efficient for the given dataset.
* RMSE for Ridge Regression is least.